# 5주차 과제: LangGraph 인지 아키텍처로 의도 분류 + 라우팅 시스템 만들기 (수정본)

## 임베딩 차원 불일치 문제 해결
- 기존 벡터 스토어의 임베딩 차원과 현재 모델의 차원이 맞지 않는 문제 수정

## Part 1: 환경 설정 및 필요 라이브러리 설치

In [1]:
# 필요한 패키지 설치
# !pip install -q langchain langchain-openai chromadb langgraph langchain-community python-dotenv sentence-transformers

In [1]:
import os
import warnings
from dotenv import load_dotenv
from typing import Dict, List, TypedDict, Annotated, Optional
from operator import add
import json

warnings.filterwarnings("ignore")

# .env 파일에서 환경 변수 로드
load_dotenv()

# OpenAI API 키 확인
if not os.getenv("OPENAI_API_KEY"):
    print(" * 경고: OPENAI_API_KEY가 .env 파일에 설정되지 않았습니다.")
else:
    print(" * OpenAI API 키 로드 완료!")

# 파일 경로 설정 (새로운 경로 사용하여 충돌 방지)
DATA_DIRECTORY = r"D:\data"
PERSIST_DIRECTORY = r"D:\data\chroma_db_langgraph"  # 새로운 경로

print("환경 설정 완료")
print(f"- 데이터 디렉토리: {DATA_DIRECTORY}")
print(f"- 벡터 DB 경로: {PERSIST_DIRECTORY}")

 * OpenAI API 키 로드 완료!
환경 설정 완료
- 데이터 디렉토리: D:\data
- 벡터 DB 경로: D:\data\chroma_db_langgraph


## Part 2: 상태(State) 정의

In [2]:
from typing import List, TypedDict
from langchain_core.messages import BaseMessage


# ChatState 클래스 정의
class ChatState(TypedDict):
    messages: List[BaseMessage]
    intent: str
    query: str
    context: str
    chat_history: List[BaseMessage]

## Part 3: LLM 및 벡터 스토어 초기화 (차원 문제 해결)

In [3]:
from langchain_openai import ChatOpenAI

# LLM 초기화
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3, max_tokens=1000)

print("LLM 초기화 완료")

LLM 초기화 완료


## 방법 1: HuggingFace 임베딩 사용 (768차원)

In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# HuggingFace 임베딩 사용 (768차원)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print("HuggingFace 임베딩 모델 로드 완료 (768차원)")

C:\Users\hfdt\AppData\Local\Temp\ipykernel_14324\3845792635.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HuggingFace 임베딩 모델 로드 완료 (768차원)


## 벡터 스토어 초기화 또는 생성

In [5]:
from langchain_community.vectorstores import Chroma
from langchain.schema import Document
import os

# 기존 벡터 스토어가 있으면 로드, 없으면 새로 생성
if os.path.exists(PERSIST_DIRECTORY) and os.listdir(PERSIST_DIRECTORY):
    # 기존 벡터 스토어 로드 시도
    try:
        vectorstore = Chroma(
            persist_directory=PERSIST_DIRECTORY, embedding_function=embeddings
        )
        print("기존 벡터 스토어 로드 완료!")
        print(f"문서 수: {vectorstore._collection.count()}")
    except Exception as e:
        print(f"기존 벡터 스토어 로드 실패: {e}")
        print("새로운 벡터 스토어를 생성합니다...")
        vectorstore = None
else:
    vectorstore = None

# 벡터 스토어가 없으면 샘플 데이터로 새로 생성
if vectorstore is None:
    print("새로운 벡터 스토어 생성 중...")

    # 샘플 문서 생성
    sample_docs = [
        Document(
            page_content="딥러닝(Deep Learning)은 인공 신경망을 기반으로 하는 머신러닝의 한 분야입니다. 여러 층의 신경망을 사용하여 복잡한 패턴을 학습합니다.",
            metadata={"topic": "딥러닝"},
        ),
        Document(
            page_content="CNN(Convolutional Neural Network)은 이미지 처리에 특화된 딥러닝 모델입니다. 합성곱 층과 풀링 층을 통해 이미지의 특징을 추출합니다.",
            metadata={"topic": "CNN"},
        ),
        Document(
            page_content="RNN(Recurrent Neural Network)은 시계열 데이터 처리에 적합한 신경망입니다. 이전 시점의 정보를 기억하여 순차적인 데이터를 처리합니다.",
            metadata={"topic": "RNN"},
        ),
        Document(
            page_content="Transformer는 어텐션 메커니즘을 사용하는 딥러닝 아키텍처입니다. BERT, GPT 같은 모델의 기반이 되며 자연어 처리에 혁명을 일으켰습니다.",
            metadata={"topic": "Transformer"},
        ),
        Document(
            page_content="강화학습(Reinforcement Learning)은 에이전트가 환경과 상호작용하며 보상을 최대화하는 방법을 학습합니다. 알파고가 대표적인 예시입니다.",
            metadata={"topic": "강화학습"},
        ),
        Document(
            page_content="전이학습(Transfer Learning)은 사전 학습된 모델을 새로운 문제에 적용하는 기법입니다. 적은 데이터로도 좋은 성능을 얻을 수 있습니다.",
            metadata={"topic": "전이학습"},
        ),
        Document(
            page_content="GAN(Generative Adversarial Network)은 생성자와 판별자가 경쟁하며 학습하는 생성 모델입니다. 이미지 생성, 스타일 변환 등에 활용됩니다.",
            metadata={"topic": "GAN"},
        ),
        Document(
            page_content="LSTM(Long Short-Term Memory)은 RNN의 장기 의존성 문제를 해결한 모델입니다. 게이트 구조를 통해 중요한 정보를 오래 기억할 수 있습니다.",
            metadata={"topic": "LSTM"},
        ),
        Document(
            page_content="오토인코더(Autoencoder)는 입력을 압축했다가 복원하는 비지도 학습 모델입니다. 차원 축소, 노이즈 제거 등에 사용됩니다.",
            metadata={"topic": "오토인코더"},
        ),
        Document(
            page_content="BERT(Bidirectional Encoder Representations from Transformers)는 양방향 트랜스포머 기반 언어 모델입니다. 문맥을 이해하는 능력이 뛰어납니다.",
            metadata={"topic": "BERT"},
        ),
    ]

    # 벡터 스토어 생성
    vectorstore = Chroma.from_documents(
        documents=sample_docs, embedding=embeddings, persist_directory=PERSIST_DIRECTORY
    )

    print("샘플 문서 임베딩 및 저장 완료!")
    print(f"문서 수: {vectorstore._collection.count()}")

# Retriever 생성
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

새로운 벡터 스토어 생성 중...
샘플 문서 임베딩 및 저장 완료!
문서 수: 10


## Part 4: 노드 함수 구현

In [6]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain


# 1. 의도 분류 노드
def classify_intent(state: ChatState) -> ChatState:
    """사용자의 의도를 분류하는 노드"""

    # 최신 사용자 메시지 추출
    if state["messages"]:
        query = state["messages"][-1].content
    else:
        query = state.get("query", "")

    # 의도 분류 프롬프트
    intent_prompt = PromptTemplate(
        input_variables=["query"],
        template="""
        다음 사용자 발화를 보고, 의도를 아래 중 하나로 골라라:
        - DOC_QA: 기술 문서 관련 질의 (딥러닝, AI, 머신러닝 등 기술적 질문)
        - SUMMARY: 이전 대화 요약이나 "중요 포인트", "정리" 요청
        - SMALL_TALK: 잡담, 일반 대화 ("안녕", "나 뭐야?" 등)
        
        사용자 발화: {query}
        
        의도만 답변하시오 (DOC_QA / SUMMARY / SMALL_TALK 중 하나):
        """,
    )

    chain = LLMChain(llm=llm, prompt=intent_prompt)
    intent_result = chain.run(query=query).strip().upper()

    # 유효성 검증
    valid_intents = ["DOC_QA", "SUMMARY", "SMALL_TALK"]
    if intent_result not in valid_intents:
        intent_result = "SMALL_TALK"  # 기본값

    state["intent"] = intent_result
    state["query"] = query

    print(f"🔍 의도 분류: {intent_result}")
    return state

In [7]:
# 2. DOC_QA 노드 (RAG Q&A)
def doc_qa_node(state: ChatState) -> ChatState:
    """RAG를 사용한 문서 Q&A 처리"""

    query = state["query"]

    try:
        # 문서 검색
        retrieved_docs = retriever.get_relevant_documents(query)

        # 컨텍스트 생성
        if retrieved_docs:
            context = "\n\n".join([doc.page_content for doc in retrieved_docs])
            state["context"] = context
        else:
            context = "관련 문서를 찾을 수 없습니다."
            state["context"] = context

    except Exception as e:
        print(f"문서 검색 중 오류 발생: {e}")
        context = "문서 검색 중 오류가 발생했습니다."
        state["context"] = context

    # RAG 프롬프트
    qa_prompt = PromptTemplate(
        input_variables=["context", "query"],
        template="""
        다음 문서 내용을 바탕으로 질문에 답변해주세요.
        답변할 수 없는 내용이면 "문서에서 관련 정보를 찾을 수 없습니다"라고 답변하세요.
        
        문서 내용:
        {context}
        
        질문: {query}
        
        답변:
        """,
    )

    chain = LLMChain(llm=llm, prompt=qa_prompt)
    answer = chain.run(context=context, query=query)

    # 답변을 메시지에 추가
    state["messages"].append(AIMessage(content=answer))

    print(f"📚 DOC_QA 답변 생성 완료")
    return state

In [14]:
# 3. SUMMARY 노드 (요약)
def summary_node(state: ChatState) -> ChatState:
    """이전 대화 요약이나 중요 포인트 정리"""

    # 이전 대화 히스토리 가져오기
    messages = state["messages"][:-1] if len(state["messages"]) > 1 else []

    if not messages:
        summary = "아직 요약할 대화 내용이 없습니다."
    else:
        # 대화 히스토리를 텍스트로 변환
        conversation = "\n".join(
            [f"{msg.__class__.__name__}: {msg.content}" for msg in messages]
        )

        summary_prompt = PromptTemplate(
            input_variables=["conversation"],
            template="""
            다음 대화 내용을 요약하거나 중요 포인트를 정리해주세요:
            
            대화 내용:
            {conversation}
            
            중요 포인트 요약:
            """,
        )

        chain = LLMChain(llm=llm, prompt=summary_prompt)
        summary = chain.run(conversation=conversation)

    state["messages"].append(AIMessage(content=summary))

    print(f"📝 SUMMARY 생성 완료")
    return state

In [15]:
# 4. SMALL_TALK 노드 (잡담)
def small_talk_node(state: ChatState) -> ChatState:
    """일반 대화 및 잡담 처리"""

    query = state["query"]

    # 대화 히스토리 컨텍스트 구성
    recent_messages = (
        state["messages"][-5:] if len(state["messages"]) > 5 else state["messages"]
    )

    chat_prompt = PromptTemplate(
        input_variables=["query", "history"],
        template="""
        친근하고 도움이 되는 어시스턴트로서 대화해주세요.
        
        대화 히스토리:
        {history}
        
        사용자: {query}
        
        어시스턴트:
        """,
    )

    history_text = (
        "\n".join(
            [f"{msg.__class__.__name__}: {msg.content}" for msg in recent_messages[:-1]]
        )
        if len(recent_messages) > 1
        else "(첫 대화입니다)"
    )

    chain = LLMChain(llm=llm, prompt=chat_prompt)
    answer = chain.run(query=query, history=history_text)

    state["messages"].append(AIMessage(content=answer))

    print(f"💬 SMALL_TALK 답변 생성 완료")
    return state

## Part 5: LangGraph 그래프 구조 정의

In [16]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver


# 라우팅 함수 정의
def route_intent(state: ChatState) -> str:
    """의도에 따라 다음 노드 결정"""
    intent = state.get("intent", "SMALL_TALK")

    if intent == "DOC_QA":
        return "doc_qa"
    elif intent == "SUMMARY":
        return "summary"
    else:  # SMALL_TALK
        return "small_talk"

In [17]:
# StateGraph 생성
workflow = StateGraph(ChatState)

# 노드 추가
workflow.add_node("router", classify_intent)
workflow.add_node("doc_qa", doc_qa_node)
workflow.add_node("summary", summary_node)
workflow.add_node("small_talk", small_talk_node)

# 엣지 정의
workflow.set_entry_point("router")

# 조건부 엣지
workflow.add_conditional_edges(
    "router",
    route_intent,
    {"doc_qa": "doc_qa", "summary": "summary", "small_talk": "small_talk"},
)

# 각 처리 노드 → END
workflow.add_edge("doc_qa", END)
workflow.add_edge("summary", END)
workflow.add_edge("small_talk", END)

# 메모리 설정
memory = MemorySaver()

# 그래프 컴파일
app = workflow.compile(checkpointer=memory)

print("그래프 구성 완료!")

그래프 구성 완료!


## Part 6: 멀티턴 대화 테스트

In [18]:
# 대화 세션 관리를 위한 헬퍼 함수
def chat_with_bot(user_input: str, thread_id: str = "test-user-1"):
    """
    사용자 입력을 받아 봇과 대화
    """

    # 초기 상태 구성
    initial_state = {
        "messages": [HumanMessage(content=user_input)],
        "query": user_input,
        "intent": "",
        "context": "",
        "chat_history": [],
    }

    # 그래프 실행
    result = app.invoke(
        initial_state, config={"configurable": {"thread_id": thread_id}}
    )

    # 최신 AI 응답 반환
    ai_messages = [msg for msg in result["messages"] if isinstance(msg, AIMessage)]
    if ai_messages:
        return ai_messages[-1].content
    return "응답 생성 실패"

In [19]:
# 테스트 시나리오 실행
print("\n========== 테스트 시작 ==========\n")

# 시나리오 1: 잡담
user_input = "안녕!"
print(f"사용자: {user_input}")
response = chat_with_bot(user_input)
print(f"봇: {response}")
print("\n---\n")

# 시나리오 2: 문서 Q&A
user_input = "딥러닝이란 뭐야?"
print(f"사용자: {user_input}")
response = chat_with_bot(user_input)
print(f"봇: {response}")
print("\n---\n")

# 시나리오 3: 추가 문서 Q&A
user_input = "CNN에 대해서도 설명해줘"
print(f"사용자: {user_input}")
response = chat_with_bot(user_input)
print(f"봇: {response}")
print("\n---\n")

# 시나리오 4: 요약 요청
user_input = "지금까지 대화 요약해줘"
print(f"사용자: {user_input}")
response = chat_with_bot(user_input)
print(f"봇: {response}")
print("\n---\n")


========== 테스트 시작 ==========

사용자: 안녕!


C:\Users\hfdt\AppData\Local\Temp\ipykernel_14324\2837808645.py:31: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=intent_prompt)
C:\Users\hfdt\AppData\Local\Temp\ipykernel_14324\2837808645.py:32: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  intent_result = chain.run(query=query).strip().upper()


🔍 의도 분류: SMALL_TALK
💬 SMALL_TALK 답변 생성 완료
봇: 안녕하세요! 어떻게 도와드릴까요? 😊

---

사용자: 딥러닝이란 뭐야?
🔍 의도 분류: DOC_QA


C:\Users\hfdt\AppData\Local\Temp\ipykernel_14324\843826731.py:9: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retrieved_docs = retriever.get_relevant_documents(query)


📚 DOC_QA 답변 생성 완료
봇: 문서에서 관련 정보를 찾을 수 없습니다.

---

사용자: CNN에 대해서도 설명해줘
🔍 의도 분류: DOC_QA
📚 DOC_QA 답변 생성 완료
봇: CNN(Convolutional Neural Network)은 이미지 처리에 특화된 딥러닝 모델로, 합성곱 층과 풀링 층을 통해 이미지의 특징을 추출합니다.

---

사용자: 지금까지 대화 요약해줘
🔍 의도 분류: SUMMARY
📝 SUMMARY 생성 완료
봇: 아직 요약할 대화 내용이 없습니다.

---

